# SmolDocling-256M-preview — DIMER document page → DocTags extraction tutorial (standalone)

[![GitHub](https://img.shields.io/badge/GitHub-181717?style=flat&logo=github&logoColor=white)](https://github.com/kurtvalcorza/smoldocling-document-extraction-pipeline) [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kurtvalcorza/smoldocling-document-extraction-pipeline/blob/main/tutorials/smoldocling_document_extraction_colab.ipynb) [![Hugging Face](https://img.shields.io/badge/%F0%9F%A4%97%20Hugging%20Face-docling--project%2FSmolDocling--256M--preview-ffcc4d?style=flat)](https://huggingface.co/docling-project/SmolDocling-256M-preview) [![Upstream](https://img.shields.io/badge/Upstream-docling--project%2Fdocling-181717?style=flat&logo=github&logoColor=white)](https://github.com/docling-project/docling) [![arXiv](https://img.shields.io/badge/arXiv-2503.11576-b31b1b.svg)](https://arxiv.org/abs/2503.11576)

**Profile:** `TASK-INFERENCE`  
**Mode:** `GUIDED`  
**Notebook specification:** DIMER Notebook Specification 2.0 — **standalone** (§4)  
**Capability:** document page image → DocTags markup (layout elements with location tokens, reading order, OCR text and OTSL tables) under one of the seven supported instructions, using the pinned `docling-project/SmolDocling-256M-preview` weights

**This notebook is standalone.** It carries the repository's pipeline module (`src/smoldocling_document_extraction_pipeline/pipeline.py` at revision `fc78b6415c23`) verbatim in Section 2, the pinned model identity and the per-file SHA-256 manifest in Section 3, and the exact runtime pins in Section 1, so it keeps working after export even if the repository changes or disappears. Its only external dependencies are the pinned PyPI distributions and the Hugging Face Hub at the immutable revision `ce51f56c4ebe36e0b1c3a55f67b261ba22a50bf8` (~518 MB, digest-verified before loading). It was generated by `tools/build_notebook.py` (build_notebook.py/2); edit the repository and regenerate rather than editing cells.

**Run all:** Selecting **Run all** in a fresh supported runtime installs the pinned dependencies, stages and digest-verifies the pinned snapshot, obtains the tutorial sample automatically, validates it into an input manifest before the model runs, runs the task locally in this kernel, writes the evaluation report, and exports machine-readable outputs with provenance. The default path needs no repository clone, no DIMER worker or service, no credential, no upload dialog and no configuration edit (NOTEBOOK_SPEC 2.0 §5).

**Bring Your Own Data:** After the sample workflow completes, set `USE_BYOD = True` in the sample cell and re-run from that cell to supply your own input. It passes through the same notebook-local validation, task, evaluation-report and export cells as the sample; the expected input format, the ceilings and the privacy guidance are stated in the Prerequisites and in the sample cell, and the upload stays inside this runtime. BYOD is optional and never part of the default path.

At inference the 256M-parameter vision–language model (a SigLIP vision encoder feeding a SmolLM-2 decoder in the Idefics3 arrangement) reads one page image — resized so its longest edge is 2048 px and split into 512-px tiles of 64 visual tokens each plus one global view — together with one instruction wrapped in the snapshot's chat template, and generates **DocTags**: a markup in which every element (`<section_header_level_1>`, `<text>`, `<otsl>` table, `<picture>`, `<caption>`, …) is preceded by four `<loc_N>` tokens on a 0–500 grid and OTSL tables carry their cells as `<ched>`/`<fcel>`/`<nl>` tokens. Decoding is greedy (`do_sample=False`) up to a caller-owned `max_new_tokens` budget. **No adaptation occurs:** no training, fine-tuning, in-context conditioning, or preprocessing fitting happens in this notebook — the upstream checkpoint supplies the weights, processor and chat template, and the carried module adds snapshot verification, the input contract (a supported instruction, image side ceilings, the token budget), a fixed output contract, and the `doctags_summary`, `doctags_to_text`, `word_error_rate`, `validate_inputs` and `evaluation_report` helpers. The default sample is a report page rendered in code from known text, so its words and its element counts serve as references; the resulting word error rate and count comparison are demonstration (plumbing) evidence for one page, not a document-conversion benchmark.

**Learning objectives:** install the pinned runtime, read what the carried pipeline module guarantees, resolve and digest-verify the immutable upstream model revision, render a synthetic report page with known reference text and element counts (or upload your own page image) and validate it into an input manifest, choose a supported instruction and a token budget, run the supported task, read DocTags correctly (elements, location tokens, OTSL cells, the `truncated` flag), recover the plain text with `doctags_to_text`, exercise an optional BYOD path, produce an evaluation report that is `sample-sanity` with `word_error_rate` and `element_count` measures only when references exist and `not-measurable` otherwise, and export the DocTags, the recovered text, an annotated page and provenance.

**This notebook does not demonstrate:** conversion of PDFs or multi-page documents (one page image per call; rendering a PDF to images is the caller's step), export to Markdown/HTML/JSON documents (that is `docling_core`'s `DoclingDocument.load_from_doctags`, not installed here), batch or streaming generation, sampling or beam search, instructions other than the seven the upstream README lists, OCR accuracy or layout/table-structure evaluation on a labelled page set (which this repository does not ship), and any training. The model is a **preview** release trained on rendered documents; scans, photographs, handwriting and non-Latin scripts are outside what this notebook measures, and its output can be truncated, repeated or hallucinated without any signal in the markup itself.

## Prerequisites

- **Runtime:** a fresh supported runtime (Google Colab or Jupyter, Python 3.12). The default path runs on CPU (float32) and uses CUDA automatically when available (bfloat16). CPU is adequate but slow: the repository's model card records 6.3 s to load and 21.7 s for one 850×1100 page (630 generated tokens) in the Windows venv (Intel Core Ultra 9 275HX); a denser page or a larger `max_new_tokens` budget scales roughly with the tokens generated. The pinned `torch==2.14.0` install and the 513 MB checkpoint are the large downloads of the run.
- **Knowledge:** basic Python and PIL; what a vision–language model's generated tokens are; what word error rate measures; that markup well-formedness is not correctness.
- **Data:** the default sample is a deterministic 850×1100 report page rendered in code with Pillow's bundled font — a heading, two paragraphs, an 8×5 and a 5×3 ruled table and a closing sentence — so nothing is downloaded and no private data is needed. Optional BYOD upload is gated off by default so the sample path can run top-to-bottom without interaction. Expected BYOD input: one image decodable by Pillow (PNG/JPEG/WebP and similar) of a **single document page**, any colour mode, sides between 16 and 4096 px. Do not upload confidential or restricted data to a hosted notebook environment unless you are authorized to do so. Uploaded inputs remain in the notebook runtime; this pipeline does not send them to a third-party inference API.
- **External access:** the Hugging Face Hub only, to fetch the pinned `docling-project/SmolDocling-256M-preview` snapshot (~518 MB in total) at revision `ce51f56c4ebe…`. No GitHub access and no credentials are required; nothing is installed from this repository.

## 1. Install the pinned runtime

The dependency set is pinned exactly (the same `==` pins as the repository's `pyproject.toml` at the generating revision) and installed directly — there is no repository clone and no package install. If a pin replaces a distribution this runtime has already imported, the cell stops with a restart instruction rather than continuing with mixed versions. Look for a dictionary reporting the notebook's source revision, Python, `torch`, `transformers` versions, and whether CUDA is available.

In [ ]:
import importlib
import importlib.metadata
import os
import platform
import subprocess
import sys

PINS = [
    'torch==2.14.0',
    'transformers==4.57.6',
    'safetensors==0.8.0',
    'numpy==2.5.3',
    'pillow==11.3.0',
    'huggingface-hub==0.36.2',
]
NOTEBOOK_SOURCE = {
    'repository': 'smoldocling-document-extraction-pipeline',
    'repository_revision': 'fc78b6415c2379bddd9d7a2b7f35bf3eb27a51b9',
    'embedded_module': 'src/smoldocling_document_extraction_pipeline/pipeline.py',
    'embedded_modules': ['src/smoldocling_document_extraction_pipeline/pipeline.py'],
    'module_sha256': '865f53f3bbe57b6da5d4da3e1409cb18b39ec0a68138b05fefd391539ed0028b',
    'generator': 'build_notebook.py/2',
    'notebook_spec': '2.0',
}
SKIP_INSTALL = os.environ.get('DIMER_NOTEBOOK_CI_PREINSTALLED') == '1'

def _installed_version(distribution):
    try:
        return importlib.metadata.version(distribution)
    except importlib.metadata.PackageNotFoundError:
        return None

if not SKIP_INSTALL:
    # Capture every distribution already imported in this runtime, whatever its module name
    # (PIL -> pillow), so a pinned install that replaces a loaded package is detected and the
    # notebook stops with a restart instruction instead of continuing with mixed versions.
    _module_dists = importlib.metadata.packages_distributions()
    _loaded = sorted({d for m in list(sys.modules) for d in _module_dists.get(m.partition('.')[0], ())})
    loaded = {distribution: _installed_version(distribution) for distribution in _loaded}
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *PINS], check=True)
    importlib.invalidate_caches()
    stale = []
    for distribution, before in loaded.items():
        installed = _installed_version(distribution)
        if before is not None and before != installed:
            stale.append(f'{distribution}: loaded={before}, installed={installed}')
    if stale:
        raise RuntimeError('Core dependencies changed while older modules were loaded: ' + '; '.join(stale) + '. Restart the runtime, then rerun from the top.')

import torch, transformers
print({'notebook_source': NOTEBOOK_SOURCE, 'python': platform.python_version(), 'torch': torch.__version__, 'transformers': transformers.__version__, 'cuda': torch.cuda.is_available()})

## 2. Pipeline code (carried verbatim from `src/smoldocling_document_extraction_pipeline/` @ `fc78b6415c23`)

The next 1 cell(s) **are** the repository's package, module by module in dependency order: the pinned identity constants, snapshot verification (`verify_snapshot`), staged download (`stage_missing_files`), the named operational ceilings, the public validation and evaluation helpers, and the pipeline class. The text is the modules', byte for byte, except for the rewrite rules listed in `tools/build_notebook.py` (1 rule(s), plus the removal of package-relative `from .x import` lines, whose names are already defined by the preceding cells). The repository's parity test (`tests/test_notebook_parity.py`) fails whenever these cells and the modules diverge, so what you run here is what the repository tests. Nothing in these cells runs a model yet.

**Module 1/1:** `src/smoldocling_document_extraction_pipeline/pipeline.py`

In [ ]:
"""Document-to-DocTags conversion with the pinned ``docling-project/SmolDocling-256M-preview`` checkpoint.

The class loads the processor and model only from a digest-verified local snapshot (``weights/<key>/``)
or, when explicitly allowed, from the Hugging Face Hub at the pinned revision — always with
``trust_remote_code=False``: the Idefics3 architecture comes from the pinned ``transformers`` release,
the weights are SafeTensors, and no model-repository code is executed.
"""

from __future__ import annotations

import hashlib
import json
import re
from collections import Counter
from collections.abc import Callable, Mapping, Sequence
from dataclasses import dataclass
from pathlib import Path
from typing import Any

from PIL import Image

MODEL_ID = "docling-project/SmolDocling-256M-preview"
MODEL_REVISION = "ce51f56c4ebe36e0b1c3a55f67b261ba22a50bf8"
MODEL_LICENSE = "cdla-permissive-2.0"
MODEL_KEY = "smoldocling-256m-preview"
DEFAULT_WEIGHTS_DIR = Path.cwd() / "weights" / MODEL_KEY  # standalone rewrite (build_notebook.py): working-directory-relative
MANIFEST_NAME = "dimer-base-manifest.json"

# The instructions the pinned README's "Supported Instructions" table lists. Any other instruction
# is refused: the model was trained on these forms and a paraphrase is undefined behaviour.
INSTRUCTIONS = (
    "Convert this page to docling.",
    "Convert chart to table.",
    "Convert formula to LaTeX.",
    "Convert code to text.",
    "Convert table to OTSL.",
    "Find all 'text' elements on the page, retrieve all section headers.",
    "Detect footer elements on the page.",
)
DEFAULT_INSTRUCTION = INSTRUCTIONS[0]
# Generation ceilings. 8192 is the max_new_tokens the pinned README's transformers example passes and
# the text model's max_position_embeddings (config.json); the default is a practical page budget.
MAX_NEW_TOKENS = 8192
DEFAULT_MAX_NEW_TOKENS = 2048
DECODING = "greedy"
# Input ceilings. The processor resizes so the longest edge is 2048 px and splits the page into
# 512-px tiles of 64 visual tokens each plus one global view (preprocessor_config.json), so image
# cost is bounded; the side ceiling only guards memory during decoding and resizing.
MAX_IMAGE_SIDE = 4096
MIN_IMAGE_SIDE = 16
# Tokens the decoder emits around the answer; stripped from the returned DocTags (the README example
# decodes with skip_special_tokens=False so the DocTags markup survives, then removes these).
_TERMINATORS = ("<end_of_utterance>", "<|im_end|>")
# DocTags element tags counted by doctags_summary (added_tokens.json names them; the loc grid is
# 0..500 per axis, four <loc_N> tokens per element box).
_ELEMENT_TAGS = (
    "section_header_level_1",
    "section_header_level_2",
    "section_header_level_3",
    "text",
    "paragraph",
    "list_item",
    "ordered_list",
    "unordered_list",
    "otsl",
    "picture",
    "caption",
    "formula",
    "code",
    "page_header",
    "page_footer",
    "footnote",
    "chart",
    "key_value_region",
)
_TAG_RE = re.compile(r"</?([a-z_]+(?:_[0-9]+)?)>")
_LOC_RE = re.compile(r"<loc_[0-9]+>")
_OTSL_CELL_RE = re.compile(r"<(?:fcel|ecel|ched|rhed|srow|lcel|ucel|xcel|nl)>")


def _sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with open(path, "rb") as fh:
        for chunk in iter(lambda: fh.read(1 << 20), b""):
            digest.update(chunk)
    return digest.hexdigest()


def verify_snapshot(path: str | Path | None = None) -> dict[str, Any]:
    """Check a local snapshot against its DIMER manifest; raise naming the first mismatch."""
    root = Path(path) if path is not None else DEFAULT_WEIGHTS_DIR
    manifest_path = root / MANIFEST_NAME
    if not manifest_path.is_file():
        raise FileNotFoundError(f"manifest not found: {manifest_path}")
    with open(manifest_path, encoding="utf-8") as fh:
        manifest = json.load(fh)
    if manifest.get("modelId") != MODEL_ID:
        raise ValueError(f"manifest modelId {manifest.get('modelId')!r} != {MODEL_ID!r}")
    if manifest.get("revision") != MODEL_REVISION:
        raise ValueError(f"manifest revision {manifest.get('revision')!r} != {MODEL_REVISION!r}")
    for entry in manifest["files"]:
        file_path = root / entry["path"]
        if not file_path.is_file():
            raise FileNotFoundError(f"snapshot file missing: {file_path}")
        size = file_path.stat().st_size
        if size != entry["bytes"]:
            raise ValueError(f"{entry['path']}: size {size} != manifest {entry['bytes']}")
        digest = _sha256(file_path)
        if digest != entry["sha256"]:
            raise ValueError(f"{entry['path']}: sha256 {digest} != manifest {entry['sha256']}")
    return {
        "path": str(root),
        "model_id": manifest["modelId"],
        "revision": manifest["revision"],
        "files": len(manifest["files"]),
        "total_bytes": manifest.get("totalBytes"),
    }


def _hub_download(relative_path: str, root: Path) -> None:
    """Fetch one manifest-listed file at MODEL_REVISION straight into the snapshot directory."""
    from huggingface_hub import hf_hub_download

    hf_hub_download(MODEL_ID, relative_path, revision=MODEL_REVISION, local_dir=str(root))


def stage_missing_files(
    path: str | Path | None = None,
    *,
    allow_download: bool = False,
    downloader: Callable[[str, Path], None] | None = None,
) -> list[str]:
    """Fetch manifest-listed files that are absent locally (a fresh clone commits the manifest but
    git-ignores the weights). Returns the relative paths fetched; `verify_snapshot` still runs after."""
    root = Path(path) if path is not None else DEFAULT_WEIGHTS_DIR
    manifest_path = root / MANIFEST_NAME
    if not manifest_path.is_file():
        raise FileNotFoundError(f"manifest not found: {manifest_path}")
    with open(manifest_path, encoding="utf-8") as fh:
        manifest = json.load(fh)
    if manifest.get("modelId") != MODEL_ID or manifest.get("revision") != MODEL_REVISION:
        raise ValueError(
            f"manifest names {manifest.get('modelId')}@{manifest.get('revision')}, "
            f"package pins {MODEL_ID}@{MODEL_REVISION}; refusing to stage"
        )
    missing = [entry["path"] for entry in manifest["files"] if not (root / entry["path"]).is_file()]
    if not missing:
        return []
    if not allow_download:
        raise FileNotFoundError(
            f"snapshot at {root} is missing {missing}; "
            f"pass allow_download=True to fetch them at {MODEL_REVISION}"
        )
    fetch = downloader or _hub_download
    for relative_path in missing:
        fetch(relative_path, root)
    return missing


def build_messages(instruction: str) -> list[dict[str, Any]]:
    """One user turn: an image placeholder then the instruction, in the snapshot chat-template shape."""
    return [{"role": "user", "content": [{"type": "image"}, {"type": "text", "text": instruction}]}]


def doctags_to_text(doctags: str) -> str:
    """Plain text carried by a DocTags string: tags and <loc_N> tokens removed, whitespace collapsed.

    OTSL table cells become space-separated words; structure is lost. This is the text a caller would
    compare with an OCR reference, not a document export (use docling_core for that).
    """
    text = _LOC_RE.sub(" ", doctags)
    text = _OTSL_CELL_RE.sub(" ", text)
    text = _TAG_RE.sub(" ", text)
    return " ".join(text.split())


def doctags_summary(doctags: str) -> dict[str, Any]:
    """Count the DocTags elements in a conversion: which structure the model claims the page has.

    Counts opening tags per element type, the number of <loc_N> tokens (four per located element),
    whether the string is wrapped in <doctag>…</doctag>, and how many OTSL table cells appear.
    """
    opened = Counter(
        match.group(1) for match in _TAG_RE.finditer(doctags) if not match.group(0).startswith("</")
    )
    counts = {tag: opened.get(tag, 0) for tag in _ELEMENT_TAGS}
    return {
        "counts": counts,
        "n_elements": sum(counts.values()),
        "n_loc_tokens": len(_LOC_RE.findall(doctags)),
        "n_table_cells": len(_OTSL_CELL_RE.findall(doctags)),
        "wrapped_in_doctag": doctags.lstrip().startswith("<doctag>")
        and doctags.rstrip().endswith("</doctag>"),
        "n_chars": len(doctags),
    }


def _tokens(text: str) -> list[str]:
    return text.lower().split()


def word_error_rate(reference: str, hypothesis: str) -> float:
    """Word error rate of ``hypothesis`` against ``reference`` after lower-casing and whitespace tokenisation.

    Levenshtein edits over words divided by reference words; punctuation is **not** stripped, so a
    stray comma counts. The metric a caller would use to score ``doctags_to_text`` against a known page.
    """
    ref, hyp = _tokens(reference), _tokens(hypothesis)
    if not ref:
        raise ValueError("reference must contain at least one word")
    previous = list(range(len(hyp) + 1))
    for row_index, ref_token in enumerate(ref, 1):
        current = [row_index]
        for column_index, hyp_token in enumerate(hyp, 1):
            current.append(
                min(
                    current[-1] + 1,
                    previous[column_index] + 1,
                    previous[column_index - 1] + (ref_token != hyp_token),
                )
            )
        previous = current
    return previous[-1] / len(ref)


def validate_image(image: Any) -> Image.Image:
    if not isinstance(image, Image.Image):
        raise TypeError(f"image must be a PIL.Image.Image, got {type(image).__name__}")
    width, height = image.size
    if min(width, height) < MIN_IMAGE_SIDE:
        raise ValueError(f"image side {min(width, height)} px < MIN_IMAGE_SIDE {MIN_IMAGE_SIDE}")
    if max(width, height) > MAX_IMAGE_SIDE:
        raise ValueError(f"image side {max(width, height)} px > MAX_IMAGE_SIDE {MAX_IMAGE_SIDE}")
    return image.convert("RGB")


INPUT_SCHEMA: dict[str, Any] = {
    "input": "one page image as PIL.Image.Image (any mode, converted to RGB) plus one supported instruction",
    "image_side_px": [MIN_IMAGE_SIDE, MAX_IMAGE_SIDE],
    "instructions": list(INSTRUCTIONS),
    "max_new_tokens": [1, MAX_NEW_TOKENS],
    "decoding": f"{DECODING} (do_sample=False), deterministic on a fixed device and dtype",
    "preprocessing": (
        "image converted to RGB; the processor resizes so the longest edge is 2048 px (aspect ratio "
        "preserved) and splits it into 512-px tiles of 64 visual tokens each plus one global view; the "
        "instruction is wrapped in the snapshot's chat template as one user turn (see build_messages)"
    ),
    "output": "DocTags markup (docling_core-compatible), decoded without dropping the tag tokens",
}


def _check_inputs(image: Any, instruction: Any, max_new_tokens: Any) -> tuple[Image.Image, str, int]:
    """Raise TypeError/ValueError naming the first violated ceiling; return the checked request.

    ``convert`` and ``validate_inputs`` both route through this function so their acceptance
    criteria cannot diverge.
    """
    rgb = validate_image(image)
    if not isinstance(instruction, str):
        raise TypeError("instruction must be a str")
    if instruction not in INSTRUCTIONS:
        raise ValueError(f"instruction {instruction!r} is not one of the supported INSTRUCTIONS")
    if isinstance(max_new_tokens, bool) or not isinstance(max_new_tokens, int):
        raise TypeError("max_new_tokens must be an int")
    if not 1 <= max_new_tokens <= MAX_NEW_TOKENS:
        raise ValueError(f"max_new_tokens must be between 1 and MAX_NEW_TOKENS={MAX_NEW_TOKENS}")
    return rgb, instruction, max_new_tokens


def validate_inputs(
    image: Image.Image,
    *,
    instruction: str = DEFAULT_INSTRUCTION,
    max_new_tokens: int = DEFAULT_MAX_NEW_TOKENS,
    names: Sequence[str] | None = None,
) -> dict[str, Any]:
    """Validation stage: return the input manifest (schema, observations, request, verdict).

    Rejection is reported by raising exactly as ``convert`` would; a caller that wants the finding
    recorded catches the exception and stores ``str(exc)`` under ``findings``.
    """
    _rgb, checked_instruction, checked_tokens = _check_inputs(image, instruction, max_new_tokens)
    if names is not None and len(names) != 1:
        raise ValueError("names must have exactly one entry (convert takes one page image)")
    return {
        "schema": dict(INPUT_SCHEMA),
        "inputs": [{"id": names[0] if names else "image-0", "mode": image.mode, "size": list(image.size)}],
        "instruction": checked_instruction,
        "generation": {"max_new_tokens": checked_tokens, "do_sample": False, "decoding": DECODING},
        "verdict": "accepted",
        "findings": [],
        "model_id": MODEL_ID,
        "model_revision": MODEL_REVISION,
    }


def evaluation_report(
    result: Mapping[str, Any],
    reference_text: str | None = None,
    expected_counts: Mapping[str, int] | None = None,
    *,
    sample_kind: str = "synthetic",
) -> dict[str, Any]:
    """Evaluation stage: a machine-readable report even when nothing is measurable.

    With ``reference_text`` (the words the page really carries) the report carries ``word_error_rate``
    of ``doctags_to_text`` against it; with ``expected_counts`` (element tag -> expected number) it
    carries one ``element_count`` entry per tag comparing expected and observed. Either makes the
    verdict ``sample-sanity``; without both it is ``not-measurable`` and the report says what labelled
    data would make the task measurable.
    """
    doctags = str(result["doctags"])
    summary = doctags_summary(doctags)
    base = {
        "task": "document page image -> DocTags (layout, reading order, OCR, tables)",
        "score_semantics": (
            "generated markup carries no score, no probability and no correctness signal; well-formed "
            "tags are not evidence that the text or layout is right. Greedy decoding makes the output "
            "reproducible on a fixed device and dtype, which is a reproducibility property, not a quality one"
        ),
        "instruction": result.get("instruction"),
        "sample_kind": sample_kind,
        "doctags_summary": summary,
        "truncated": result.get("truncated"),
        "baselines": [],
        "model_id": MODEL_ID,
        "model_revision": MODEL_REVISION,
    }
    metrics: list[dict[str, Any]] = []
    if reference_text:
        metrics.append(
            {
                "id": "word_error_rate",
                "value": word_error_rate(reference_text, doctags_to_text(doctags)),
                "normalisation": (
                    "lower-cased, whitespace-tokenised, tags and <loc_N> removed; punctuation kept"
                ),
                "estimation": "one page, no dispersion estimate",
            }
        )
    for tag, expected in (expected_counts or {}).items():
        if tag not in _ELEMENT_TAGS:
            raise ValueError(f"unknown element tag {tag!r}; expected one of {_ELEMENT_TAGS}")
        metrics.append(
            {
                "id": "element_count",
                "tag": tag,
                "expected": int(expected),
                "observed": summary["counts"][tag],
                "estimation": "one page, structural sanity only",
            }
        )
    if not metrics:
        return {
            **base,
            "metrics": [],
            "verdict": "not-measurable",
            "reason": "no reference text or expected element counts were supplied for the evaluated page",
            "needs": (
                "pages with ground-truth text and layout (for example DocLayNet-style annotations or the "
                "publisher's source) scored with word_error_rate on the OCR and with layout/table metrics "
                "such as TEDS on the structure; no such labelled set ships with this repository"
            ),
        }
    return {
        **base,
        "metrics": metrics,
        "verdict": "sample-sanity",
        "reason": (
            f"{len(metrics)} sanity measure(s) on one tutorial page whose text and layout you rendered "
            "yourself; plumbing evidence, not a document-conversion benchmark"
        ),
        "needs": (
            "a labelled page set from the deployment domain (scans, publishers, layouts, tables) for any "
            "OCR accuracy, layout or table-structure claim"
        ),
    }


@dataclass
class SmolDoclingPipeline:
    """``_runner(image, instruction, max_new_tokens)`` returns ``{"doctags": str, "new_tokens": int}``."""

    _runner: Callable[..., dict[str, Any]]
    device: str = "cpu"
    dtype: str = "float32"
    source: str = "injected"

    @classmethod
    def from_pretrained(
        cls,
        device: str | None = None,
        weights_dir: str | Path | None = None,
        allow_download: bool = False,
    ) -> SmolDoclingPipeline:
        root = Path(weights_dir or DEFAULT_WEIGHTS_DIR)
        common: dict[str, Any] = {"trust_remote_code": False}
        if (root / MANIFEST_NAME).is_file():
            stage_missing_files(root, allow_download=allow_download)
            verify_snapshot(root)
            location, common["local_files_only"], source = str(root), True, "local-snapshot"
        elif allow_download:
            location, common["revision"], source = MODEL_ID, MODEL_REVISION, "hf-hub"
        else:
            raise FileNotFoundError(
                f"no verified snapshot at {root} and allow_download=False; "
                f"stage it with: hf download {MODEL_ID} --revision {MODEL_REVISION} --local-dir {root}"
            )
        # Refuse invalid snapshots before importing model libraries.
        import torch
        from transformers import AutoModelForImageTextToText, AutoProcessor

        resolved_device = device or ("cuda:0" if torch.cuda.is_available() else "cpu")
        dtype = torch.bfloat16 if resolved_device.startswith("cuda") else torch.float32
        processor = AutoProcessor.from_pretrained(location, **common)
        model = AutoModelForImageTextToText.from_pretrained(location, dtype=dtype, **common)
        model = model.eval().to(resolved_device)

        def runner(image: Image.Image, instruction: str, max_new_tokens: int) -> dict[str, Any]:
            text = processor.apply_chat_template(build_messages(instruction), add_generation_prompt=True)
            inputs = processor(text=text, images=[image], return_tensors="pt").to(resolved_device)
            with torch.inference_mode():
                generated = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
            new_ids = generated[0, inputs["input_ids"].shape[1] :]
            # skip_special_tokens=False keeps the DocTags markup (the tags are added tokens); the
            # terminator tokens are removed afterwards, as the pinned README's example does.
            decoded = processor.batch_decode(new_ids.unsqueeze(0), skip_special_tokens=False)[0]
            return {"doctags": decoded, "new_tokens": int(new_ids.shape[0])}

        return cls(runner, resolved_device, str(dtype).removeprefix("torch."), source)

    def convert(
        self,
        image: Image.Image,
        *,
        instruction: str = DEFAULT_INSTRUCTION,
        max_new_tokens: int = DEFAULT_MAX_NEW_TOKENS,
    ) -> dict[str, Any]:
        """Run one supported instruction over one page image; ``doctags`` is the decoded markup."""
        rgb, checked_instruction, checked_tokens = _check_inputs(image, instruction, max_new_tokens)
        raw = self._runner(rgb, checked_instruction, checked_tokens)
        if not isinstance(raw, dict) or "doctags" not in raw:
            raise RuntimeError("runner must return a dict with 'doctags'")
        doctags = str(raw["doctags"])
        for terminator in _TERMINATORS:
            doctags = doctags.replace(terminator, "")
        doctags = doctags.strip()
        new_tokens = int(raw.get("new_tokens", 0))
        return {
            "doctags": doctags,
            "text": doctags_to_text(doctags),
            "instruction": checked_instruction,
            "image_size": list(rgb.size),
            "new_tokens": new_tokens,
            "truncated": new_tokens >= checked_tokens,
            "generation": {"max_new_tokens": checked_tokens, "do_sample": False, "decoding": DECODING},
            "device": self.device,
            "dtype": self.dtype,
            "source": self.source,
            "model_id": MODEL_ID,
            "model_revision": MODEL_REVISION,
        }

## 3. Pin, stage and verify the model

The model identity is carried twice — `MODEL_ID`/`MODEL_REVISION` in the module above and the `13`-file manifest below (paths, byte sizes, SHA-256) — and the cell first asserts they agree. It writes the manifest into the working-directory snapshot, then `stage_missing_files(..., allow_download=True)` fetches exactly the entries that are absent from the Hugging Face Hub **at revision `ce51f56c4ebe…`** (never `main`), `verify_snapshot` re-hashes every file and raises on the first size or digest mismatch, and only then does `SmolDoclingPipeline.from_pretrained(weights_dir=WEIGHTS_DIR)` load the verified files. There is no fallback to a different download and no remote model code is executed. The effective identity, device and weight source are printed before any inference.

In [ ]:
import json

MANIFEST = {
  "format": "dimer_hf_snapshot",
  "formatVersion": 1,
  "modelKey": "smoldocling-256m-preview",
  "modelId": "docling-project/SmolDocling-256M-preview",
  "revision": "ce51f56c4ebe36e0b1c3a55f67b261ba22a50bf8",
  "files": [
    {
      "path": "README.md",
      "bytes": 16108,
      "sha256": "9b82c4dd1b38340656da55d628d63bf319c5fc4700811148687b6d8070e7e493"
    },
    {
      "path": "added_tokens.json",
      "bytes": 3667,
      "sha256": "fc79a032b551636ad0fe6c0e16bfe38c43b5843895cbb0544a4a5919818472cc"
    },
    {
      "path": "chat_template.json",
      "bytes": 430,
      "sha256": "b585e3598909a5687f9f9d738d35223724dedef256b9b274e1cbfb32b13c74bf"
    },
    {
      "path": "config.json",
      "bytes": 3903,
      "sha256": "57af2810c65b9896a8d1d65c67aabdb9296d497aa10a6329f4bb2ddce623586f"
    },
    {
      "path": "generation_config.json",
      "bytes": 141,
      "sha256": "0758109c85e7f7d6b0202ebf643bb07c5625b3363e389854b414d9a701becc28"
    },
    {
      "path": "merges.txt",
      "bytes": 466391,
      "sha256": "0b54e8aa4e53d5383e2e4bc635a56b43f9647f7b13832d5d9ecd8f82dac4f510"
    },
    {
      "path": "model.safetensors",
      "bytes": 513028808,
      "sha256": "cdcdf5d823c5684029c7d8e52177cf10f9034b3aba6577549cfb1a9ce36ad0a2"
    },
    {
      "path": "preprocessor_config.json",
      "bytes": 486,
      "sha256": "6cb6e36d6fcb88ca1502c4a26750715dc3e7dedddc9a8f17b27d8d167d1457e7"
    },
    {
      "path": "processor_config.json",
      "bytes": 68,
      "sha256": "e7bff42da73ae9eec9042ef20e066e11f1ee20f025358ff79131e3c0fb549b46"
    },
    {
      "path": "special_tokens_map.json",
      "bytes": 1069,
      "sha256": "aa0ff906077086dfa9734a7f97f68c825877a48f9468807be65504495cdeef09"
    },
    {
      "path": "tokenizer.json",
      "bytes": 3547443,
      "sha256": "7c5cf6233a3dc8b9e54fb729ee6e771bdf5f0d65fd9075b5e60bed837959deee"
    },
    {
      "path": "tokenizer_config.json",
      "bytes": 27362,
      "sha256": "b38f39506a4fa7d2604015a6c303df320675cfeb7ba1f5969e1c44032727107b"
    },
    {
      "path": "vocab.json",
      "bytes": 800662,
      "sha256": "82b84012e3add4d01d12ba14442026e49b8cbbaead1f79ecf3d919784f82dc79"
    }
  ],
  "totalBytes": 517896538
}

if (MANIFEST['modelId'], MANIFEST['revision']) != (MODEL_ID, MODEL_REVISION):
    raise RuntimeError('inline manifest does not name the identity carried by the pipeline module; the notebook was not regenerated after a change')
WEIGHTS_DIR = DEFAULT_WEIGHTS_DIR
WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)
with open(WEIGHTS_DIR / MANIFEST_NAME, 'w', encoding='utf-8') as handle:
    json.dump(MANIFEST, handle, indent=2)
print({'model_id': MODEL_ID, 'revision': MODEL_REVISION, 'license': MODEL_LICENSE, 'files': len(MANIFEST['files']), 'total_bytes': MANIFEST['totalBytes']})
fetched = stage_missing_files(WEIGHTS_DIR, allow_download=True)
print({'weights_dir': str(WEIGHTS_DIR), 'fetched': fetched})
snapshot = verify_snapshot(WEIGHTS_DIR)
_files = snapshot.get('files', []) if isinstance(snapshot, dict) else []
print({'verified_files': len(_files) if isinstance(_files, list) else _files, 'revision': snapshot.get('revision', MODEL_REVISION) if isinstance(snapshot, dict) else MODEL_REVISION})
pipe = SmolDoclingPipeline.from_pretrained(weights_dir=WEIGHTS_DIR)
print({'device': getattr(pipe, 'device', None), 'source': getattr(pipe, 'source', 'local-snapshot')})

## 4. Render the synthetic report page or optional BYOD

The default sample is **synthetic** and carries its own references: a page with a heading, two six-line paragraphs of short words, an 8×5 and a 5×3 ruled table of short tokens and a closing sentence is rendered with Pillow's bundled font at 850×1100 — the same page the repository's smoke run used. The words drawn on it, in reading order, are the reference text for the `word_error_rate` sanity check later, and the drawn structure (1 section header, 3 text blocks, 2 tables) is the expected element count. Neither is a labelled dataset, so nothing here is an OCR or layout benchmark. The image digest is printed for the record. BYOD is optional and disabled by default; when enabled, upload one page image — no reference exists for it, so the evaluation report will be `not-measurable`.

The instruction and the token budget are **caller-owned request parameters**: `instruction` must be one of the seven forms in `INSTRUCTIONS` (the pinned README's supported-instructions table; the default converts the whole page), and `max_new_tokens` bounds the generation (`DEFAULT_MAX_NEW_TOKENS = 2048` is a practical page budget, `MAX_NEW_TOKENS = 8192` is the ceiling the README example uses). Nothing is validated in this cell — the next section hands the image and the request to the pipeline's own validation stage, which is the only checker. Look for a dictionary naming the sample kind, the page size and digest, the request, and the number of reference words.

In [ ]:
import hashlib
import io

import numpy as np
from PIL import Image, ImageDraw, ImageFont

USE_BYOD = False  # @param {type:"boolean"}
instruction = 'Convert this page to docling.'  # @param ['Convert this page to docling.', 'Convert table to OTSL.', "Find all 'text' elements on the page, retrieve all section headers.", 'Detect footer elements on the page.']
max_new_tokens = 2048  # @param {type:"integer"}


def synthetic_page(width=850, height=1100):
    """Heading, two paragraphs, an 8x5 and a 5x3 ruled table, one closing line. Returns page, reference text, expected counts."""
    page = Image.new('RGB', (width, height), 'white')
    d = ImageDraw.Draw(page)
    body, head = ImageFont.load_default(size=15), ImageFont.load_default(size=22)
    words = 'quarterly revenue by region and product line for the fiscal year with notes on methodology'.split()
    reference = ['Annual Report: Regional Results']
    d.text((70, 50), reference[0], fill='black', font=head)
    y = 95
    for _para in range(2):
        for line in range(6):
            text = ' '.join(words[(line * 3 + k) % len(words)] for k in range(11 - (line % 3)))
            d.text((70, y), text, fill=(40, 40, 40), font=body)
            reference.append(text)
            y += 20
        y += 16
    for x0, y0, x1, y1, rows, cols, first in ((70, 330, 780, 660, 8, 5, 'Region'), (70, 760, 430, 990, 5, 3, 'Item')):
        d.rectangle([x0, y0, x1, y1], outline='black', width=2)
        rh, cw = (y1 - y0) / rows, (x1 - x0) / cols
        d.line([(x0, y0 + rh), (x1, y0 + rh)], fill='black', width=2)
        for r in range(2, rows):
            d.line([(x0, y0 + rh * r), (x1, y0 + rh * r)], fill=(120, 120, 120), width=1)
        for c in range(1, cols):
            d.line([(x0 + cw * c, y0), (x0 + cw * c, y1)], fill=(120, 120, 120), width=1)
        for r in range(rows):
            for c in range(cols):
                token = (first if c == 0 else f'Q{c}') if r == 0 else (f'North {r}' if c == 0 else f'{(r * 7 + c * 13) % 97 + 1},{(r * 31 + c) % 900 + 100:03d}')
                d.text((x0 + cw * c + 8, y0 + rh * r + rh / 2 - 8), token, fill='black', font=body)
                reference.append(token)
    tail = 'Table 2 summarises the line items; see the appendix for the full breakdown.'
    d.text((70, 1010), tail, fill=(40, 40, 40), font=body)
    reference.append(tail)
    return page, ' '.join(reference), {'section_header_level_1': 1, 'text': 3, 'otsl': 2}


if USE_BYOD:
    from google.colab import files
    uploaded = files.upload()
    image_name = next(iter(uploaded))
    image = Image.open(io.BytesIO(uploaded[image_name]))
    image.load()
    reference_text, expected_counts = None, None
    sample_kind = 'BYOD'
else:
    # Deterministic synthetic page: no randomness, so no seed is needed and the digest is stable per Pillow build.
    image, reference_text, expected_counts = synthetic_page()
    image_name = 'synthetic_report_page_850x1100.png'
    sample_kind = 'synthetic'

image_sha256 = hashlib.sha256(np.asarray(image.convert('RGB')).tobytes()).hexdigest()
print({'sample_kind': sample_kind, 'name': image_name, 'mode': image.mode, 'size': image.size, 'rgb_sha256': image_sha256, 'instruction': instruction, 'max_new_tokens': max_new_tokens, 'reference_words': None if reference_text is None else len(reference_text.split()), 'expected_counts': expected_counts})

## 5. Validate the request → input manifest

`validate_inputs` is the pipeline's public validation stage: it applies exactly the checks `convert` applies — image type and sides `MIN_IMAGE_SIDE`..`MAX_IMAGE_SIDE` px, an instruction that is one of the seven `INSTRUCTIONS`, and `max_new_tokens` in `[1, MAX_NEW_TOKENS]` — and returns an **input manifest** naming the schema (including the preprocessing the processor applies and the decoding rule), the input's observed mode and size, the request, and the verdict. The manifest is written to `outputs/smoldocling_document_extraction_input_manifest.json`. To show what rejection looks like, the cell also validates a paraphrased instruction the model was not trained on and records the pipeline's own error message as a finding. Inside the pipeline the image is converted to RGB and tiled by the processor; nothing else is dropped or altered. The pipeline cannot tell whether the image is a document page: that contract is the caller's.

In [ ]:
import json
import os

os.makedirs('outputs', exist_ok=True)
print({'ceilings': {'MIN_IMAGE_SIDE': MIN_IMAGE_SIDE, 'MAX_IMAGE_SIDE': MAX_IMAGE_SIDE, 'MAX_NEW_TOKENS': MAX_NEW_TOKENS, 'DEFAULT_MAX_NEW_TOKENS': DEFAULT_MAX_NEW_TOKENS, 'DECODING': DECODING, 'INSTRUCTIONS': list(INSTRUCTIONS)}})
input_manifest = validate_inputs(image, instruction=instruction, max_new_tokens=max_new_tokens, names=[image_name])
# Demonstrate rejection on a request that breaks the contract; the finding is recorded, not swallowed.
try:
    validate_inputs(image, instruction='Please convert this page to markdown.')
except ValueError as exc:
    input_manifest['findings'].append({'input': 'unsupported-instruction-probe', 'verdict': 'rejected', 'message': str(exc)})
with open('outputs/smoldocling_document_extraction_input_manifest.json', 'w', encoding='utf-8') as handle:
    json.dump(input_manifest, handle, indent=2, ensure_ascii=False)
print(json.dumps(input_manifest, indent=2))

## 6. Convert the page and read DocTags correctly

`convert` returns a dict with `doctags` — the generated markup with the terminator tokens removed — `text` (the plain words `doctags_to_text` recovers: tags, `<loc_N>` and OTSL cell tokens stripped, whitespace collapsed), the `instruction`, `image_size`, `new_tokens`, a `truncated` flag that is true when the budget was exhausted, the generation settings and the model identity. **No score exists**: generated markup carries no probability and no correctness signal, and well-formed tags are not evidence that the text or the layout is right. `doctags_summary` counts the opening tags per element type, the `<loc_N>` tokens (four per located element), the OTSL cell tokens and whether the string is wrapped in `<doctag>…</doctag>`. Greedy decoding is deterministic on a fixed device and dtype; CUDA kernel selection and bfloat16 on GPU can change a token and therefore the rest of the sequence, so GPU and CPU outputs need not match. As recorded in the model card, the repository's CPU smoke on this same page generated 630 tokens in 21.7 s and counted exactly 1 `section_header_level_1`, 3 `text` and 2 `otsl` elements with 68 OTSL cell tokens — both tables cell-perfect, the paragraphs with three short spans dropped and one span repeated, and the heading without its colon; that is one observation on one rendered page, not a calibration point.

In [ ]:
import time

t0 = time.time()
result = pipe.convert(image, instruction=instruction, max_new_tokens=max_new_tokens)
elapsed = time.time() - t0
summary = doctags_summary(result['doctags'])
print({'seconds': round(elapsed, 1), 'new_tokens': result['new_tokens'], 'truncated': result['truncated'], 'device': pipe.device, 'dtype': pipe.dtype, 'n_elements': summary['n_elements'], 'n_table_cells': summary['n_table_cells'], 'wrapped_in_doctag': summary['wrapped_in_doctag']})
print({tag: n for tag, n in summary['counts'].items() if n})
print(result['doctags'][:1200] + ('…' if len(result['doctags']) > 1200 else ''))
print('--- recovered text ---')
print(result['text'][:600] + ('…' if len(result['text']) > 600 else ''))
if result['truncated']:
    print('The token budget was exhausted: the DocTags are incomplete. Raise max_new_tokens (ceiling MAX_NEW_TOKENS) and rerun.')

## 7. Evaluate → evaluation report

`evaluation_report` is the pipeline's public evaluation stage and always produces a report. No conversion metric is reported by default: OCR accuracy, layout mAP or table TEDS need labelled pages, and this repository ships none. The repository's metric helpers are `word_error_rate` (lower-cased, whitespace-tokenised Levenshtein distance over words, punctuation kept) and the element-count comparison; when reference text is supplied the report carries one `word_error_rate` entry, and when expected counts are supplied one `element_count` entry per tag comparing expected and observed, with the verdict `sample-sanity`. On the synthetic path those references are words and structure **you rendered yourself**, so a low error rate proves only that the input contract, forward pass, decoding and text recovery round-trip. On BYOD no reference exists, the verdict is `not-measurable`, and the report states what would make the task measurable. The report is written to `outputs/smoldocling_document_extraction_evaluation_report.json`.

In [ ]:
report = evaluation_report(result, reference_text, expected_counts, sample_kind=sample_kind)
with open('outputs/smoldocling_document_extraction_evaluation_report.json', 'w', encoding='utf-8') as handle:
    json.dump(report, handle, indent=2, ensure_ascii=False)
print(json.dumps({k: v for k, v in report.items() if k not in ('metrics', 'doctags_summary')}, indent=2))
for metric in report['metrics']:
    if metric['id'] == 'word_error_rate':
        print(f"word_error_rate {metric['value']:.3f}  ({metric['normalisation']})")
    else:
        print(f"element_count {metric['tag']:24} expected {metric['expected']:>3}  observed {metric['observed']:>3}")
if report['verdict'] == 'not-measurable':
    print('No reference text or expected counts exist for this input, so nothing is measured; inspect the annotated PNG and the recovered text instead.')

## 8. Export outputs and provenance

Machine-readable JSON preserves the full result (DocTags, recovered text, the request, `new_tokens`, `truncated`), the element summary, the evaluation report, the input manifest, the sample identity, digest and references, the notebook's source (repository, revision, embedded module digest, generator), the model identifier, the immutable model revision, the model licence, and the runtime identity (Python, `torch`, `transformers`, device). The DocTags are also written verbatim to `outputs/smoldocling_document_extraction.dt` (the file form `docling_core` loads) and the recovered text to `outputs/smoldocling_document_extraction.txt`, and an annotated PNG draws each located element's `<loc_N>` box (0–500 grid scaled to the page; headers in red, tables in green, everything else in blue) for visual inspection — a supplement to, not a replacement for, the machine-readable files. No credentials are recorded.

In [ ]:
import re

LOC_ELEMENT = re.compile(r'<([a-z_0-9]+)><loc_(\d+)><loc_(\d+)><loc_(\d+)><loc_(\d+)>')
COLOURS = {'section_header_level_1': (200, 30, 30), 'section_header_level_2': (200, 30, 30), 'otsl': (0, 160, 0)}
annotated = image.convert('RGB').copy()
draw = ImageDraw.Draw(annotated)
width, height = annotated.size
located = []
for match in LOC_ELEMENT.finditer(result['doctags']):
    tag, x0, y0, x1, y1 = match.group(1), *(int(v) for v in match.groups()[1:])
    box = [x0 / 500 * width, y0 / 500 * height, x1 / 500 * width, y1 / 500 * height]
    located.append({'tag': tag, 'box': [round(v, 1) for v in box]})
    draw.rectangle(box, outline=COLOURS.get(tag, (40, 90, 220)), width=2)
annotated.save('outputs/smoldocling_document_extraction_annotated.png')
with open('outputs/smoldocling_document_extraction.dt', 'w', encoding='utf-8') as handle:
    handle.write(result['doctags'])
with open('outputs/smoldocling_document_extraction.txt', 'w', encoding='utf-8') as handle:
    handle.write(result['text'])
payload = {
    'prediction': result,
    'doctags_summary': summary,
    'located_elements': located,
    'evaluation_report': report,
    'input_manifest': input_manifest,
    'sample': {'kind': sample_kind, 'name': image_name, 'size': list(image.size), 'rgb_sha256': image_sha256, 'reference_text': reference_text, 'expected_counts': expected_counts},
    'notebook_source': NOTEBOOK_SOURCE,
    'repository_revision': NOTEBOOK_SOURCE['repository_revision'],
    'model_id': MODEL_ID,
    'model_revision': MODEL_REVISION,
    'model_license': MODEL_LICENSE,
    'runtime': {
        'python': platform.python_version(),
        'torch': torch.__version__,
        'transformers': transformers.__version__,
        'device': pipe.device,
        'dtype': pipe.dtype,
    },
}
with open('outputs/smoldocling_document_extraction_result.json', 'w', encoding='utf-8') as handle:
    json.dump(payload, handle, indent=2, ensure_ascii=False)
print({'located_elements': len(located)})
print(sorted(os.listdir('outputs')))

## Interpretation and limits

The DocTags are the structure and text the model *claims* the page has; nothing in the markup scores that claim, and a well-formed document with plausible tables can still carry wrong words, merged lines, repeated phrases or invented elements. On the synthetic page the `word_error_rate` and `element_count` entries in the evaluation report compare the output with words and structure you rendered yourself and the verdict is `sample-sanity`, which proves only that the input contract, forward pass, decoding and text recovery work (the repository's smoke run measured a word error rate of 0.19 on this page, with both tables cell-perfect and the errors in the running text); they say nothing about scans, photographs, dense multi-column layouts, formulas, code, non-Latin scripts or long pages, and a BYOD result is a single-page observation with the verdict `not-measurable`. **The model generates markup for any image** and stops only at a terminator or the token budget: check `truncated`, expect repetition loops on inputs unlike its training renders, and treat the `<loc_N>` boxes as approximate (a 0–500 grid, about 2 px per step on this page). The pipeline provides no PDF rendering, no document export, no batch generation, no evaluation on labelled pages and no training capability.

Successful execution proves that the recorded repository revision's pipeline module, carried in this notebook, can acquire and digest-verify the pinned model, validate the demonstrated request, execute the public pipeline path, and emit the shown machine-readable outputs in the tested runtime — without the repository being reachable. It does **not** establish benchmark superiority, deployment calibration, safety for high-consequence decisions, or production fitness on an unseen domain.

**Next experiments:** switch `instruction` to `Convert table to OTSL.` and crop the page to the first table to see a bare `<otsl>` answer; lower `max_new_tokens` to 200 and watch `truncated` turn true; enable `USE_BYOD` with a page whose text you know, pass that text as `reference_text` to `evaluation_report` and see the verdict switch to `sample-sanity`; then install `docling_core` and load `outputs/smoldocling_document_extraction.dt` with `DoclingDocument.load_from_doctags` to export Markdown.

## References

- Repository README: https://github.com/kurtvalcorza/smoldocling-document-extraction-pipeline/blob/main/README.md
- Repository model card: https://github.com/kurtvalcorza/smoldocling-document-extraction-pipeline/blob/main/MODEL_CARD.md
- Weight provenance: https://github.com/kurtvalcorza/smoldocling-document-extraction-pipeline/blob/main/docs/WEIGHTS.md
- Upstream model: https://huggingface.co/docling-project/SmolDocling-256M-preview
- Upstream code (Docling): https://github.com/docling-project/docling
- SmolDocling: An ultra-compact vision-language model for end-to-end multi-modal document conversion (Nassar et al., 2025): https://arxiv.org/abs/2503.11576
- Docling Technical Report (Auer et al., 2024): https://arxiv.org/abs/2408.09869